# Keypoint RCNN-Pose Evaluation | Keypoint Detection | Stomata (4 KP)

**Model**: Keypoint R CNN-Pose-X101 (finetuned)
**Task**: Keypoint Detection
**Eval**: COCO OKS-based Keypoint AP + Bbox AP

This notebook reproduces the Keypoint R-CNN evaluation reported in the CVPRW stomata keypoint benchmark.
It is intended for AP-based benchmark evaluation only.
Some public benchmark splits contain annotations only; for those splits, users must separately download the original images from the source links listed in the Hugging Face dataset card.

---
## 1. Configuration & Environment

In [1]:
import sys
import platform
import logging
import warnings
import pandas as pd
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")
print("=" * 60)
print("ENVIRONMENT INFO")
print("=" * 60)
print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print()

ENVIRONMENT INFO
Python: 3.12.2 | packaged by conda-forge | (main, Feb 16 2024, 20:50:58) [GCC 12.3.0]
Platform: Linux-5.14.0-162.6.1.el9_1.x86_64-x86_64-with-glibc2.34



In [2]:
# Core packages
import torch
import numpy as np
import cv2
from PIL import Image

print("=" * 60)
print("CORE PACKAGE VERSIONS")
print("=" * 60)
print(f"torch:        {torch.__version__}")
print(f"numpy:        {np.__version__}")
print(f"opencv (cv2): {cv2.__version__}")

import PIL; print(f"Pillow:       {PIL.__version__}")
print()

CORE PACKAGE VERSIONS
torch:        2.7.0+cu126
numpy:        1.26.0
opencv (cv2): 4.9.0
Pillow:       11.1.0



In [3]:
# Detectron2
import detectron2
from detectron2 import __version__ as d2_version

print("=" * 60)
print("DETECTRON2 PACKAGE VERSIONS")
print("=" * 60)
print(f"detectron2:   {d2_version}")

# Check CUDA availability
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version:   {torch.version.cuda}")
    print(f"cuDNN version:  {torch.backends.cudnn.version()}")
    print(f"GPU:            {torch.cuda.get_device_name(0)}")
print()

DETECTRON2 PACKAGE VERSIONS
detectron2:   0.6

CUDA available: True
CUDA version:   12.6
cuDNN version:  90501
GPU:            NVIDIA A100 80GB PCIe



In [ ]:
# Evaluation packages
import pycocotools
import scipy
import json
from pathlib import Path
from importlib.metadata import version

print("=" * 60)
print("EVALUATION PACKAGE VERSIONS")
print("=" * 60)
pycocotools_version = version('pycocotools')
print(f"pycocotools: {pycocotools_version}")
print(f"scipy:        {scipy.__version__}")
print()

EVALUATION PACKAGE VERSIONS
pycocotools: 2.0.10
scipy:        1.12.0



In [ ]:
# ======================== CONFIGURATION ========================
# Add the dataset and model files from the hugging face links in your custom directory
# Some benchmark splits in the public dataset release contain annotations only.
# For those splits, users must separately download the original images from the source links listed in the hugging face dataset card 
# Place them in the expected folder structure before running evaluation.

from pathlib import Path
import numpy as np
import torch
import os

MODEL_NAME = "KP-RCNN-X101"
TASK = 'keypoint_detection'
KRCNN_WEIGHTS = Path(os.environ.get("KRCNN_WEIGHTS", "weights/KP-RCNN-X101/model_final.pth"))
#https://huggingface.co/Sainath001/stomata-keypoint-benchmark-cvpr-agrivision-2026-models/tree/main/KP-RCNN-X101
SCORE_THRESHOLD = 0.3
MAX_DETS_PER_IMAGE = 300
STOMATA_CAT_ID = 1
NUM_KEYPOINTS = 4
OKS_SIGMAS = np.array([0.1, 0.1, 0.1, 0.1], dtype=np.float32)
DATA_ROOT = Path(os.environ.get("DATA_ROOT", "data/stomata-keypoint-benchmark/datasets"))
#https://huggingface.co/datasets/Sainath001/stomata-keypoint-benchmark-cvpr-agrivision-2026-Dataset/tree/main/datasets
KRCNN_CONFIG_PATH = os.environ.get("KRCNN_CONFIG_PATH", "")


def _has_images(d: Path) -> bool:
    exts = (".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp")
    for ext in exts:
        if any(d.glob(f"*{ext}")):
            return True
    return False

def build_datasets(root: str) -> dict:
    root = Path(root)
    datasets = {}

    for ds_dir in sorted(root.iterdir()):
        if not ds_dir.is_dir():
            continue
        gt_json = ds_dir / "coco.json"
        if not gt_json.exists():
            jsons = list(ds_dir.glob("*.json"))
            if len(jsons) == 1:
                gt_json = jsons[0]
            else:
                continue
        images_dir = ds_dir / "images"
        if images_dir.is_dir() and _has_images(images_dir):
            images_path = images_dir
        else:
            images_path = ds_dir
        key = ds_dir.name
        datasets[key] = {
            "gt_json": str(gt_json),
            "images":  str(images_path),
        }

    return datasets

DATASETS = build_datasets(DATA_ROOT)

# Output directory for prediction JSONs
RESULTS_ROOT = Path("artifacts") / MODEL_NAME
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
PRED_ROOT = RESULTS_ROOT / "preds"
PRED_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

print(f"Model:           {MODEL_NAME}")
print(f"Task:            {TASK}")
print(f"Score threshold: {SCORE_THRESHOLD}")
print(f"Max dets/image:  {MAX_DETS_PER_IMAGE}")
print(f"Num keypoints:   {NUM_KEYPOINTS}")
print(f"OKS sigmas:      {OKS_SIGMAS}")
print(f"Datasets:        {list(DATASETS.keys())}")
print(f'Weights:         {KRCNN_WEIGHTS}')
print(f"Pred dir:        {PRED_ROOT}")
print(f"Device:          {DEVICE}")

---
## 2. Generate COCO-format Keypoint Predictions

In [6]:
# ======================== UTILITY FUNCTIONS ========================

from typing import List, Dict, Tuple

def xyxy_to_xywh(box_xyxy: np.ndarray) -> List[float]:
    """Convert [x1, y1, x2, y2] to COCO [x, y, w, h]."""
    x1, y1, x2, y2 = [float(v) for v in box_xyxy]
    return [x1, y1, max(0.0, x2 - x1), max(0.0, y2 - y1)]


def clip_xywh(x, y, w, h, W, H):
    """Clip box [x, y, w, h] to image bounds."""
    x = max(0.0, min(float(x), W - 1.0))
    y = max(0.0, min(float(y), H - 1.0))
    w = max(0.0, min(float(w), W - x))
    h = max(0.0, min(float(h), H - y))
    return x, y, w, h


def load_coco_items(gt_json, images_root):
    """Load COCO GT and return (coco_obj, items_list, size_map)."""
    coco = COCO(gt_json)
    imgs = coco.loadImgs(coco.getImgIds())
    items = []
    size_map = {}
    images_root = Path(images_root)
    for im in imgs:
        iid = int(im['id'])
        H, W = int(im['height']), int(im['width'])
        fp = images_root / im['file_name']
        if not fp.exists():
            fp = images_root / Path(im['file_name']).name
        if fp.exists():
            items.append((iid, str(fp), (H, W)))
            size_map[iid] = (H, W)
    return coco, items, size_map

In [ ]:
# ======================== KEYPOINT R-CNN (Detectron2) INFERENCE ========================
import os
from pathlib import Path
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor
import detectron2
from tqdm.auto import tqdm as tqdm_fn
import torch

# Base Detectron2 Keypoint R-CNN config to instantiate the architecture
# The original fine-tuning config file was not preserved with the checkpoint.
# This notebook reconstructs the architecture from the standard Detectron2
# Keypoint R-CNN X101 config and then loads the fine-tuned weights.
BASE_MODEL = "COCO-Keypoints/keypoint_rcnn_X_101_32x8d_FPN_3x.yaml" 
BASE_WEIGHTS = (
    "detectron2://COCO-Keypoints/keypoint_rcnn_X_101_32x8d_FPN_3x/"
    "139686956/model_final_5ad38f.pkl"
)

def find_d2_config(rel_path: str) -> str:
    """Find Detectron2 config file on disk by searching common locations.
    rel_path example:
      "COCO-Keypoints/keypoint_rcnn_X_101_32x8d_FPN_3x.yaml"
    """
    local_try = Path("configs") / rel_path
    if local_try.exists():
        return str(local_try.resolve())

    d2_root = Path(detectron2.__file__).resolve().parent
    search_roots = [
        d2_root,
        d2_root.parent,          
        d2_root.parent.parent,   
    ]

    target_tail = Path(rel_path)

    for root in search_roots:
        direct = [
            root / "model_zoo" / "configs" / target_tail,
            root / "configs" / target_tail,
            root / "detectron2" / "model_zoo" / "configs" / target_tail,
        ]
        for p in direct:
            if p.exists():
                return str(p)

        max_depth = 3
        root = root.resolve()
        for dirpath, dirnames, filenames in os.walk(root):
            dp = Path(dirpath)
            rel_depth = len(dp.relative_to(root).parts)
            if rel_depth > max_depth:
                dirnames[:] = []  
                continue
            if dp.name == "configs" and (dp / target_tail).exists():
                return str((dp / target_tail).resolve())

    raise FileNotFoundError(
        f"Could not locate Detectron2 config: {rel_path}.\n"
        f"Fix options:\n"
        f"  (A) Copy Detectron2 configs folder locally and set ./configs/{rel_path}\n"
        f"  (B) Provide the yaml path directly by setting KRCNN_CONFIG_PATH\n"
    )

def build_kprcnn_predictor(
    weights_path: str,
    score_thr: float = 0.3,
    device: str = "cuda:0",
    max_dets: int = 300,
) -> DefaultPredictor:
    cfg = get_cfg()
    cfg_path = None
    if "KRCNN_CONFIG_PATH" in globals() and isinstance(KRCNN_CONFIG_PATH, str) and len(KRCNN_CONFIG_PATH.strip()) > 0:
        cfg_path = KRCNN_CONFIG_PATH.strip()
    else:
        cfg_path = find_d2_config(BASE_MODEL)

    cfg.merge_from_file(cfg_path)
    conv_dim = None
    try:
        ckpt = torch.load(weights_path, map_location='cpu')
        state = ckpt.get('model', ckpt)
        w = state.get('roi_heads.keypoint_head.conv_fcn1.weight', None)
        if w is not None:
            conv_dim = int(w.shape[0])
    except Exception as _e:
        conv_dim = None
    if conv_dim is not None:
        cfg.MODEL.ROI_KEYPOINT_HEAD.CONV_DIMS = tuple([conv_dim] * 8)
        print(f"[INFO] Setting ROI_KEYPOINT_HEAD.CONV_DIMS to {cfg.MODEL.ROI_KEYPOINT_HEAD.CONV_DIMS} (from checkpoint)")
    if Path(weights_path).exists():
        cfg.MODEL.WEIGHTS = weights_path
    else:
        cfg.MODEL.WEIGHTS = BASE_WEIGHTS
        print(f"[WARN] KRCNN_WEIGHTS not found at: {weights_path}")
        print(f"       Falling back to Detectron2 pretrained weights: {BASE_WEIGHTS}")

    cfg.MODEL.DEVICE = device
    cfg.MODEL.KEYPOINT_ON = True
    cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1
    cfg.MODEL.ROI_KEYPOINT_HEAD.NUM_KEYPOINTS = NUM_KEYPOINTS

    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = float(score_thr)
    cfg.TEST.DETECTIONS_PER_IMAGE = int(max_dets)
    print(f"KPRCNN config:   {cfg_path}")
    print(f"KPRCNN weights:  {cfg.MODEL.WEIGHTS}")
    return DefaultPredictor(cfg)

def infer_kprcnn_pose(
    weights_path: str,
    items: List[Tuple[int, str, Tuple[int, int]]],
    device: str = "cuda:0",
    score_thr: float = 0.3,
    max_dets: int = 300,
    num_kp: int = 4,
) -> List[Dict]:
    """Run Keypoint R-CNN inference and return COCO-format keypoint predictions."""
    predictor = build_kprcnn_predictor(
        weights_path=weights_path,
        score_thr=score_thr,
        device=device,
        max_dets=max_dets,
    )

    preds = []
    for image_id, path, (H, W) in tqdm_fn(items, desc="KPRCNN Inference"):
        img = cv2.imread(path)
        if img is None:
            continue

        out = predictor(img)
        inst = out["instances"].to("cpu")
        if len(inst) == 0:
            continue

        boxes_xyxy = inst.pred_boxes.tensor.numpy()  # (N,4)
        scores = inst.scores.numpy()                # (N,)

        if not hasattr(inst, "pred_keypoints"):
            continue
        kps = inst.pred_keypoints.numpy()           

        for i in range(len(inst)):
            s = float(scores[i])
            if s < score_thr:
                continue

            bbox_xywh = xyxy_to_xywh(boxes_xyxy[i])
            x, y, w, h = clip_xywh(*bbox_xywh, W, H)
            if w <= 0 or h <= 0:
                continue

            kp = kps[i]  # (K,3)
            coco_kp = []
            for ki in range(num_kp):
                kx, ky, kp_prob = float(kp[ki, 0]), float(kp[ki, 1]), float(kp[ki, 2])
                v = 2 if kp_prob > 0 else 0
                coco_kp.extend([kx, ky, v])

            preds.append({
                "image_id": int(image_id),
                "category_id": STOMATA_CAT_ID,
                "bbox": [x, y, w, h],
                "score": s,
                "keypoints": coco_kp,
            })

    print(f"Total predictions: {len(preds)}")
    return preds

In [ ]:
# ======================== RUN INFERENCE ON ALL SPLITS ========================
from typing import List, Dict, Tuple

# COCO API
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

pred_files = {}  # {split_name: pred_json_path}

for ds_name, ds_cfg in DATASETS.items():
    print(f"\n{'=' * 60}")
    print(f'Dataset: {ds_name}')
    print(f"{'=' * 60}")
    
    out_json = PRED_ROOT / f'{ds_name}_kp_preds.json'
    if out_json.exists():
        print(f'  Predictions already exist: {out_json}')
        print(f'  Delete file to regenerate.')
        pred_files[ds_name] = str(out_json)
        continue
    
    coco_gt, items, size_map = load_coco_items(ds_cfg['gt_json'], ds_cfg['images'])
    print(f'  Images found: {len(items)}')
    
    preds = infer_kprcnn_pose(
        weights_path=KRCNN_WEIGHTS,
        items=items,
        device=DEVICE,
        score_thr=SCORE_THRESHOLD,
        max_dets=MAX_DETS_PER_IMAGE,
        num_kp=NUM_KEYPOINTS,
    )
    
    with open(out_json, 'w') as f:
        json.dump(preds, f, indent=2)
    print(f'  Saved {len(preds)} predictions → {out_json}')
    pred_files[ds_name] = str(out_json)

print(f"\n{'=' * 60}")
print('PREDICTION FILES:')
for name, path in pred_files.items():
    print(f'  {name}: {path}')

---
## 3. COCO Keypoint Evaluation (OKS-based AP)

In [9]:
# ======================== KEYPOINT EVALUATION ========================

def run_coco_keypoint_eval(
    gt_json: str,
    pred_json: str,
    sigmas: np.ndarray = None,
) -> Dict:
    """
    Run COCO keypoint evaluation using OKS.
    
    Args:
        gt_json:   Path to COCO ground truth JSON (with keypoint annotations)
        pred_json: Path to COCO prediction JSON (with keypoints)
        sigmas:    OKS sigma per keypoint (controls tolerance)
    
    Returns:
        Dict with AP, AP50, AP75, AR metrics
    """
    coco_gt = COCO(gt_json)
    coco_gt.dataset.setdefault('info', {})
    coco_gt.dataset.setdefault('licenses', []) 
    coco_dt = coco_gt.loadRes(pred_json)
    
    coco_eval = COCOeval(coco_gt, coco_dt, iouType='keypoints')
    
    # Override default COCO sigmas with our stomata-specific sigmas
    if sigmas is not None:
        coco_eval.params.kpt_oks_sigmas = sigmas

    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()
    
    s = coco_eval.stats
    return {
        'KP_AP':       s[0],
        'KP_AP50':     s[1],
        'KP_AP75':     s[2],
        'KP_AP_M':     s[3],
        'KP_AP_L':     s[4],
        'KP_AR':       s[5],
        'KP_AR50':     s[6],
        'KP_AR75':     s[7],
        'KP_AR_M':     s[8],
        'KP_AR_L':     s[9],
    }

In [10]:
# ======================== RUN KEYPOINT EVAL ========================

kp_results = {}

for ds_name, ds_cfg in DATASETS.items():
    print(f"\n{'=' * 60}")
    print(f'Keypoint Eval: {ds_name}')
    print(f"{'=' * 60}")
    
    pred_json = pred_files.get(ds_name)
    if pred_json is None or not Path(pred_json).exists():
        print(f'  Prediction file not found!')
        continue
    
    try:
        metrics = run_coco_keypoint_eval(
            gt_json=ds_cfg['gt_json'],
            pred_json=pred_json,
            sigmas=OKS_SIGMAS,
        )
        kp_results[ds_name] = metrics
    except Exception as e:
        print(f'  Error: {e}')
        import traceback; traceback.print_exc()

# Display
if kp_results:
    print(f"\n{'=' * 60}")
    print(f'KEYPOINT AP RESULTS — {MODEL_NAME} (OKS sigmas={OKS_SIGMAS[0]})')
    print(f"{'=' * 60}")
    df_kp = pd.DataFrame(kp_results).T
    df_kp = (df_kp * 100).round(2)
    display(df_kp)


Keypoint Eval: T_HR5MZ
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *keypoints*
DONE (t=0.42s).
Accumulating evaluation results...
DONE (t=0.00s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.037
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.085
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.008
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets= 20 ] = 0.029
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets= 20 ] = 0.087
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.061
 Average Recall     (AR) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.115
 Average Recall     (AR) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.030
 Average Recall     (AR) @[ IoU=0.50

,KP_AP,KP_AP50,KP_AP75,KP_AP_M,KP_AP_L,KP_AR,KP_AR50,KP_AR75,KP_AR_M,KP_AR_L
T_HR5MZ,3.67,8.54,0.77,2.85,8.74,6.11,11.47,3.03,4.12,14.83
T_HR5WH,15.86,33.21,11.66,13.99,18.24,22.29,39.87,20.26,18.21,23.68
T_MSAA,3.34,9.51,1.39,3.34,-100.00,5.72,14.19,2.70,5.72,-100.00
T_MZA,31.82,49.16,36.28,31.82,-100.00,36.85,50.36,42.26,36.85,-100.00
T_MZLP,27.86,48.79,29.04,27.86,0.00,33.30,50.41,37.58,33.31,0.00
T_NKBY,0.00,0.00,0.00,-100.00,0.00,0.00,0.00,0.00,-100.00,0.00
T_SBGH,0.00,0.00,0.00,0.00,-100.00,0.00,0.00,0.00,0.00,-100.00
T_TCAB,0.01,0.12,0.00,0.00,0.02,0.07,0.75,0.00,0.00,0.08
T_TCMZ,6.89,14.11,3.59,0.41,7.53,15.78,28.15,10.37,2.50,16.84


---
## 4. COCO Bbox Evaluation

In [11]:
# ======================== BBOX EVALUATION ========================

def run_coco_bbox_eval(gt_json: str, pred_json: str) -> Dict:
    """Run standard COCO bbox evaluation."""
    coco_gt = COCO(gt_json)
    coco_gt.dataset.setdefault('info', {})
    coco_gt.dataset.setdefault('licenses', []) 
    coco_dt = coco_gt.loadRes(pred_json)
    
    coco_eval = COCOeval(coco_gt, coco_dt, iouType='bbox')
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()
    
    s = coco_eval.stats
    return {
        'Box_AP':     s[0],
        'Box_AP50':   s[1],
        'Box_AP75':   s[2],
        'Box_AP_S':   s[3],
        'Box_AP_M':   s[4],
        'Box_AP_L':   s[5],
        'Box_AR@1':   s[6],
        'Box_AR@10':  s[7],
        'Box_AR@100': s[8],
    }


bbox_results = {}

for ds_name, ds_cfg in DATASETS.items():
    print(f"\n{'=' * 60}")
    print(f'Bbox Eval: {ds_name}')
    print(f"{'=' * 60}")
    
    pred_json = pred_files.get(ds_name)
    if pred_json is None or not Path(pred_json).exists():
        print(f'  Prediction file not found!')
        continue
    
    try:
        metrics = run_coco_bbox_eval(
            gt_json=ds_cfg['gt_json'],
            pred_json=pred_json,
        )
        bbox_results[ds_name] = metrics
    except Exception as e:
        print(f'  Error: {e}')

# Display
if bbox_results:
    print(f"\n{'=' * 60}")
    print(f'BBOX AP RESULTS — {MODEL_NAME}')
    print(f"{'=' * 60}")
    df_box = pd.DataFrame(bbox_results).T
    df_box = (df_box * 100).round(2)
    display(df_box)


Bbox Eval: T_HR5MZ
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.52s).
Accumulating evaluation results...
DONE (t=0.00s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.310
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.550
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.309
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.268
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.524
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.007
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.074
 Average Recall     (AR) @[ IoU=0.50:0.95 | 

,Box_AP,Box_AP50,Box_AP75,Box_AP_S,Box_AP_M,Box_AP_L,Box_AR@1,Box_AR@10,Box_AR@100
T_HR5MZ,30.99,54.98,30.86,-100.0,26.78,52.36,0.75,7.40,34.69
T_HR5WH,21.44,55.81,6.63,-100.0,19.72,22.40,2.03,16.14,25.42
T_MSAA,12.72,29.43,6.71,-100.0,12.72,-100.00,1.37,12.23,15.65
T_MZA,47.39,93.65,38.56,-100.0,47.39,-100.00,1.61,15.90,56.23
T_MZLP,40.35,90.75,25.21,-100.0,40.35,10.00,1.36,14.60,49.61
T_NKBY,9.63,30.69,0.93,-100.0,-100.00,9.68,2.11,14.57,14.57
T_SBGH,0.00,0.00,0.00,0.0,0.00,-100.00,0.00,0.00,0.00
T_TCAB,0.88,1.91,0.00,-100.0,0.00,1.27,0.90,1.27,1.27
T_TCMZ,41.89,69.20,48.00,-100.0,14.68,44.80,6.48,46.07,49.15


---
## 5. Summary

In [12]:
# ======================== COMBINED SUMMARY ========================

print(f"\n{'=' * 60}")
print(f'SUMMARY — {MODEL_NAME}')
print(f'Confidence threshold: {SCORE_THRESHOLD}')
print(f'OKS sigmas: {OKS_SIGMAS.tolist()}')
print(f"{'=' * 60}")

summary_rows = []
for ds_name in DATASETS:
    row = {'Split': ds_name}
    if ds_name in kp_results:
        row.update(kp_results[ds_name])
    if ds_name in bbox_results:
        row.update(bbox_results[ds_name])
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows).set_index('Split')

# Show key columns
key_cols = ['Box_AP', 'Box_AP50', 'KP_AP', 'KP_AP50', 'KP_AP75', 'KP_AR']
key_cols = [c for c in key_cols if c in df_summary.columns]
df_key = (df_summary[key_cols] * 100).round(2)

print('\nKey Metrics (%):')
display(df_key)


SUMMARY — KRCNN-101
Confidence threshold: 0.3
OKS sigmas: [0.10000000149011612, 0.10000000149011612, 0.10000000149011612, 0.10000000149011612]

Key Metrics (%):


,Box_AP,Box_AP50,KP_AP,KP_AP50,KP_AP75,KP_AR
Split,,,,,,
T_HR5MZ,30.99,54.98,3.67,8.54,0.77,6.11
T_HR5WH,21.44,55.81,15.86,33.21,11.66,22.29
T_MSAA,12.72,29.43,3.34,9.51,1.39,5.72
T_MZA,47.39,93.65,31.82,49.16,36.28,36.85
T_MZLP,40.35,90.75,27.86,48.79,29.04,33.30
T_NKBY,9.63,30.69,0.00,0.00,0.00,0.00
T_SBGH,0.00,0.00,0.00,0.00,0.00,0.00
T_TCAB,0.88,1.91,0.01,0.12,0.00,0.07
T_TCMZ,41.89,69.20,6.89,14.11,3.59,15.78


In [ ]:
# ======================== SAVE ALL RESULTS ========================

results_output = {
    'model': MODEL_NAME,
    'task': TASK,
    'score_threshold': SCORE_THRESHOLD,
    'oks_sigmas': OKS_SIGMAS.tolist(),
    'num_keypoints': NUM_KEYPOINTS,
    'keypoint_eval': kp_results,
    'bbox_eval': bbox_results,
}

results_json = PRED_ROOT / 'evaluation_results.json'
with open(results_json, 'w') as f:
    json.dump(results_output, f, indent=2, default=float)
print(f'Results saved to: {results_json}')

Expected local layout
data/stomata-keypoint-benchmark/
├── KP-Train/
├── T-MZA/
├── T-MZLP/
├── ...
weights/KP-RCNN-X101/
└── model_final.pth